# Task 5: Dataset Selection
## Token Classification: POS Tagging & Chunking
### Overview
- This project demonstrates fine-tuning a transformer model (DistilBERT) for Part-of-Speech (POS) tagging using the Universal Dependencies dataset.
Tuning BERT for POS Tagging & Chunking


In [1]:
# Install required libraries 
!pip install transformers datasets evaluate seqeval -q

In [16]:
# Imports
import numpy as np
from datasets import load_dataset
import evaluate

from transformers import DataCollatorForTokenClassification
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

In [17]:
import sys
!{sys.executable} -m pip install conllu

### Task 1: Dataset Selection

#### Dataset Info

- Dataset: Universal Dependencies (en_ewt)
- Task: POS Tagging
- Labels: NOUN, VERB, ADJ, ADV, PRON, DET, ADP, etc.

In [18]:
# Load dataset (Universal Dependencies - POS tagging)
dataset = load_dataset("universal_dependencies", "en_ewt", trust_remote_code=True)

# Checking dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2077
    })
})


In [19]:
# Label names (POS tags)
label_list = dataset["train"].features["upos"].feature.names
print(label_list)

['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


### Task 2: Data Preprocessing

In [20]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [21]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [22]:
# Apply preprocessing
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2077/2077 [00:00<00:00, 45293.02 examples/s]


### Task 3: Model Setup

#### Model
- Model: DistilBERT (distilbert-base-uncased)
- Task Type: Token Classification
- Library: Hugging Face Transformers

In [23]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 6437.62it/s]
DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Task 4: Training

In [24]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    
    report_to=[],              
    disable_tqdm=True         
)

### Task 5: Evaluation 

In [25]:
metric = evaluate.load("seqeval")

In [26]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [27]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [28]:
trainer.train()

{'loss': '2.22', 'grad_norm': '4.275', 'learning_rate': '1.804e-05', 'epoch': '0.2'}
{'loss': '0.9059', 'grad_norm': '3.127', 'learning_rate': '1.604e-05', 'epoch': '0.4'}
{'loss': '0.4419', 'grad_norm': '3.411', 'learning_rate': '1.404e-05', 'epoch': '0.6'}
{'loss': '0.3337', 'grad_norm': '3.13', 'learning_rate': '1.204e-05', 'epoch': '0.8'}
{'loss': '0.2778', 'grad_norm': '2.433', 'learning_rate': '1.004e-05', 'epoch': '1'}
{'loss': '0.2215', 'grad_norm': '3.975', 'learning_rate': '8.04e-06', 'epoch': '1.2'}
{'loss': '0.2161', 'grad_norm': '2.682', 'learning_rate': '6.04e-06', 'epoch': '1.4'}
{'loss': '0.2015', 'grad_norm': '3.521', 'learning_rate': '4.04e-06', 'epoch': '1.6'}
{'loss': '0.176', 'grad_norm': '2.219', 'learning_rate': '2.04e-06', 'epoch': '1.8'}
{'loss': '0.1713', 'grad_norm': '2.526', 'learning_rate': '4e-08', 'epoch': '2'}


Writing model shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.10it/s]


{'train_runtime': '56', 'train_samples_per_second': '71.43', 'train_steps_per_second': '8.928', 'train_loss': '0.5166', 'epoch': '2'}


TrainOutput(global_step=500, training_loss=0.516622995376587, metrics={'train_runtime': 56.0022, 'train_samples_per_second': 71.426, 'train_steps_per_second': 8.928, 'train_loss': 0.516622995376587, 'epoch': 2.0})

In [29]:
trainer.evaluate()

/opt/anaconda3/envs/py311/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '0.2527', 'eval_precision': '0.928', 'eval_recall': '0.9192', 'eval_f1': '0.9236', 'eval_runtime': '1.556', 'eval_samples_per_second': '321.4', 'eval_steps_per_second': '40.5', 'epoch': '2'}


/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/anaconda3/envs/py311/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/opt/a

{'eval_loss': 0.25268566608428955,
 'eval_precision': 0.928035132455022,
 'eval_recall': 0.9191805808895749,
 'eval_f1': 0.9235866347102777,
 'eval_runtime': 1.5555,
 'eval_samples_per_second': 321.443,
 'eval_steps_per_second': 40.502,
 'epoch': 2.0}

### Task 6: Inference

In [30]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"
result = nlp(sentence)

for r in result:
    print(r)

{'entity': 'PROPN', 'score': np.float32(0.8973015), 'index': 1, 'word': 'john', 'start': 0, 'end': 4}
{'entity': 'VERB', 'score': np.float32(0.94054735), 'index': 2, 'word': 'works', 'start': 5, 'end': 10}
{'entity': 'ADP', 'score': np.float32(0.97242075), 'index': 3, 'word': 'at', 'start': 11, 'end': 13}
{'entity': 'PROPN', 'score': np.float32(0.9524553), 'index': 4, 'word': 'google', 'start': 14, 'end': 20}
{'entity': 'ADP', 'score': np.float32(0.97375804), 'index': 5, 'word': 'in', 'start': 21, 'end': 23}
{'entity': 'PROPN', 'score': np.float32(0.96111405), 'index': 6, 'word': 'california', 'start': 24, 'end': 34}


POS Tagging:
- Assigns grammatical labels (noun, verb, adjective)
- Works at word level
- Easier task

Chunking:
- Groups words into phrases (NP, VP)
- Works at phrase level
- More complex than POS tagging